# CDSE S3 Bucket Download Tutorial (STAC API)

This tutorial demonstrates how to:
1. Search for Sentinel-1 SLC (`IW` mode, `VV`/`VH` pol) data over Vienna using the CDSE STAC API.
2. Resolve the S3 path for the product using the CDSE OData API.
3. Download the full 4-8 GB product via the CDSE S3 interface using Boto3.

### Prerequisites
Make sure your CDSE S3 keys are loaded as environment variables:
- `cdse_S3_KEY`
- `cdse_S3_SECRET`

In [1]:
import os
import time
import requests
import boto3
import concurrent.futures

# Load S3 credentials from environment variables
ACCESS_KEY = os.environ.get('cdse_S3_KEY')
SECRET_KEY = os.environ.get('cdse_S3_SECRET')

if not ACCESS_KEY or not SECRET_KEY:
    print("Warning: S3 credentials not found. Please set them before running the download step.")

In [4]:
#print(ACCESS_KEY)
#print(SECRET_KEY)

In [9]:
config = {
    "orbit_state": "ascending",
    "relative_orbit": 73,
    "start_date": "2020-06-01",
    "end_date": "2020-07-01",
    "instrument_mode": "IW",
    "vienna_bbox": [16.1840505, 48.119439, 16.578036, 48.32185013]
}

In [10]:
# set a download directory

folder_name = (
    f"S1_{config['orbit_state'][:3].upper()}_"
    f"{str(config['relative_orbit']).zfill(3)}_"
)

DOWNLOAD_PATH = f"{os.getenv('HOME')}/bucket/WP5_Infrastructure_and_Underground_Safety/InSAR-Daten/{folder_name}/"

print(DOWNLOAD_PATH)




/home/lukas-zgis---14367194/bucket/WP5_Infrastructure_and_Underground_Safety/InSAR-Daten/S1_ASC_073_/


## 1. Search for a Scene via CDSE STAC API
We query the `sentinel-1-slc` collection using a bounding box and date range, then filter the results for our specific Beam Mode (`IW`) and Polarization (`VV`).

In [11]:
stac_endpoint = "https://stac.dataspace.copernicus.eu/v1/search"

payload = {
    "collections": ["sentinel-1-slc"],
    "bbox": config["vienna_bbox"],
    "datetime": f"{config['start_date']}T00:00:00Z/{config['end_date']}T00:00:00Z",
    "limit": 100,
    "query": {
        "sat:orbit_state": {"eq": config["orbit_state"]},
        "sat:relative_orbit": {"eq": config["relative_orbit"]},
        "sar:instrument_mode": {"eq": config["instrument_mode"]}
    }
}

items = []
url = stac_endpoint

while url:
    r = requests.post(url, json=payload if url == stac_endpoint else None)
    r.raise_for_status()
    data = r.json()

    items.extend(data.get("features", []))

    next_link = next((l for l in data.get("links", []) if l["rel"] == "next"), None)
    url = next_link["href"] if next_link else None

# minimal post-filter for VV
items = [it for it in items if "VV" in it["properties"].get("sar:polarizations", [])]

print(f"\nFound {len(items)} scenes:\n")

scenes = []

for it in items:
    base_id = it["id"]
    product_name = base_id if base_id.endswith(".SAFE") else f"{base_id}.SAFE"
    p = it["properties"]
    
    scenes.append({
        "product_name": product_name,
        "datetime": p.get("datetime"),
        "orbit_state": p.get("sat:orbit_state"),
        "relative_orbit": p.get("sat:relative_orbit"),
        "instrument_mode": p.get("sar:instrument_mode"),
        "polarizations": p.get("sar:polarizations")
    })

# print everything nicely
for s in scenes:
    print(
        s["product_name"], "|",
        s["datetime"], "|",
        s["orbit_state"], "|",
        s["relative_orbit"], "|",
        s["instrument_mode"], "|",
        s["polarizations"]
    )



Found 5 scenes:

S1A_IW_SLC__1SDV_20200628T164258_20200628T164325_033220_03D93D_D0A2.SAFE | 2020-06-28T16:42:58.015791Z | ascending | 73 | IW | ['VV', 'VH']
S1B_IW_SLC__1SDV_20200622T164215_20200622T164242_022149_02A09E_A86F.SAFE | 2020-06-22T16:42:15.843687Z | ascending | 73 | IW | ['VV', 'VH']
S1A_IW_SLC__1SDV_20200616T164257_20200616T164325_033045_03D3ED_9C02.SAFE | 2020-06-16T16:42:57.416451Z | ascending | 73 | IW | ['VV', 'VH']
S1B_IW_SLC__1SDV_20200610T164215_20200610T164242_021974_029B48_9341.SAFE | 2020-06-10T16:42:15.294282Z | ascending | 73 | IW | ['VV', 'VH']
S1A_IW_SLC__1SDV_20200604T164256_20200604T164324_032870_03CEAB_C48F.SAFE | 2020-06-04T16:42:56.566226Z | ascending | 73 | IW | ['VV', 'VH']


## 2. Get the S3 Path via CDSE OData API
We use the Copernicus Data Space Ecosystem OData API to pinpoint the exact internal S3 path for the `.SAFE` product.

In [12]:
for s in scenes:
    product_name = s["product_name"]

    odata_url = (
        "https://catalogue.dataspace.copernicus.eu/odata/v1/Products"
        f"?$filter=Name eq '{product_name}'"
    )

    r = requests.get(odata_url)
    r.raise_for_status()

    product_data = r.json().get("value", [])
    
    if not product_data:
        print(f"{product_name} | NOT FOUND")
        continue

    s3_path = product_data[0]["S3Path"]

    # split into bucket + prefix
    parts = s3_path.lstrip("/").split("/", 1)
    bucket = parts[0]
    prefix = parts[1]

    # store for later use (important!)
    s["bucket"] = bucket
    s["prefix"] = prefix

    # print nicely
    print(
        #f"{product_name} | {s['datetime']} | "
        f"{bucket} | {prefix}"
    )

eodata | Sentinel-1/SAR/SLC/2020/06/28/S1A_IW_SLC__1SDV_20200628T164258_20200628T164325_033220_03D93D_D0A2.SAFE
eodata | Sentinel-1/SAR/SLC/2020/06/22/S1B_IW_SLC__1SDV_20200622T164215_20200622T164242_022149_02A09E_A86F.SAFE
eodata | Sentinel-1/SAR/SLC/2020/06/16/S1A_IW_SLC__1SDV_20200616T164257_20200616T164325_033045_03D3ED_9C02.SAFE
eodata | Sentinel-1/SAR/SLC/2020/06/10/S1B_IW_SLC__1SDV_20200610T164215_20200610T164242_021974_029B48_9341.SAFE
eodata | Sentinel-1/SAR/SLC/2020/06/04/S1A_IW_SLC__1SDV_20200604T164256_20200604T164324_032870_03CEAB_C48F.SAFE


## 3. Download from CDSE S3
Finally, we connect to `eodata.dataspace.copernicus.eu` to retrieve the files. 
*Note: Sentinel-1 SLC datasets are massive (often 4-8GB), so this step will take some time especially if you are running it locally it will depend on your connection.*

In [13]:
def download_single_file(s3_key, bucket_name, local_file_path):
    """
    Helper function to download a single file. 
    Creates a new Boto3 session to ensure thread safety.
    """
    # Skip if it's just a directory marker
    if local_file_path.endswith('/'):
        os.makedirs(local_file_path, exist_ok=True)
        return
        
    os.makedirs(os.path.dirname(local_file_path), exist_ok=True)
    
    # Boto3 sessions are NOT thread-safe, so we instantiate a new client per thread
    session = boto3.session.Session()
    s3_client = session.client(
        's3',
        endpoint_url='https://eodata.dataspace.copernicus.eu',
        aws_access_key_id=ACCESS_KEY,
        aws_secret_access_key=SECRET_KEY,
        region_name='default'
    )
    
    s3_client.download_file(bucket_name, s3_key, local_file_path)

def download_product_from_s3_parallel(bucket_name, prefix, target_dir, max_threads=8):
    # Initial resource just to list the objects
    s3_resource = boto3.resource(
        's3',
        endpoint_url='https://eodata.dataspace.copernicus.eu',
        aws_access_key_id=ACCESS_KEY,
        aws_secret_access_key=SECRET_KEY,
        region_name='default'
    )
    
    bucket = s3_resource.Bucket(bucket_name)
    objects = list(bucket.objects.filter(Prefix=prefix))
    
    if not objects:
        print("No files found for this product prefix in CDSE S3.")
        return
    
    # Calculate total size before downloading
    total_bytes = sum(obj.size for obj in objects)
    total_mb = total_bytes / (1024 * 1024)
    
    print(f"Starting parallel download of {len(objects)} files (Total size: {total_mb:.2f} MB) to '{target_dir}'...")
    print(f"Using up to {max_threads} concurrent threads.")
    
    start_time = time.time()
    
    # Use ThreadPoolExecutor for parallel downloads
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_threads) as executor:
        futures = []
        for obj in objects:
            relative_path = os.path.relpath(obj.key, prefix)
            local_file_path = os.path.join(target_dir, relative_path)
            
            # Submit task to the thread pool
            futures.append(
                executor.submit(download_single_file, obj.key, bucket_name, local_file_path)
            )
            
        # Wait for all futures to complete
        concurrent.futures.wait(futures)
            
    end_time = time.time()
    
    duration_sec = end_time - start_time
    speed_mbps = total_mb / duration_sec if duration_sec > 0 else 0
    
    print(f"\nDownload complete! Files saved in: {os.path.abspath(target_dir)}")
    print("\n--- Parallel Download Statistics ---")
    print(f"Total Data Downloaded: {total_mb:.2f} MB")
    print(f"Total Time Taken:      {duration_sec:.2f} seconds")
    print(f"Average Speed:         {speed_mbps:.2f} MB/s")



In [14]:
target_location = f"{DOWNLOAD_PATH}{scenes[0]['product_name']}"
print(target_location)

/home/lukas-zgis---14367194/bucket/WP5_Infrastructure_and_Underground_Safety/InSAR-Daten/S1_ASC_073_/S1A_IW_SLC__1SDV_20200628T164258_20200628T164325_033220_03D93D_D0A2.SAFE


In [15]:
# Execute download for all scenes
if ACCESS_KEY and SECRET_KEY:
    for s in scenes:
        bucket = s["bucket"]
        prefix = s["prefix"]
        product_name = s["product_name"]

        target_location = f"{DOWNLOAD_PATH}/{product_name}"

        print(f"\nDownloading: {product_name}")
        
        download_product_from_s3_parallel(
            bucket,
            prefix,
            target_dir=target_location,
            max_threads=8
        )
else:
    print("Skipping download. Missing S3 credentials.")


Downloading: S1A_IW_SLC__1SDV_20200628T164258_20200628T164325_033220_03D93D_D0A2.SAFE
Starting parallel download of 39 files (Total size: 8154.98 MB) to '/home/lukas-zgis---14367194/bucket/WP5_Infrastructure_and_Underground_Safety/InSAR-Daten/S1_ASC_073_//S1A_IW_SLC__1SDV_20200628T164258_20200628T164325_033220_03D93D_D0A2.SAFE'...
Using up to 8 concurrent threads.

Download complete! Files saved in: /home/lukas-zgis---14367194/bucket/WP5_Infrastructure_and_Underground_Safety/InSAR-Daten/S1_ASC_073_/S1A_IW_SLC__1SDV_20200628T164258_20200628T164325_033220_03D93D_D0A2.SAFE

--- Parallel Download Statistics ---
Total Data Downloaded: 8154.98 MB
Total Time Taken:      84.42 seconds
Average Speed:         96.60 MB/s

Downloading: S1B_IW_SLC__1SDV_20200622T164215_20200622T164242_022149_02A09E_A86F.SAFE
Starting parallel download of 39 files (Total size: 7801.75 MB) to '/home/lukas-zgis---14367194/bucket/WP5_Infrastructure_and_Underground_Safety/InSAR-Daten/S1_ASC_073_//S1B_IW_SLC__1SDV_20200